<div style='background:#1A1A2E;padding:32px;border-radius:10px;margin-bottom:20px'>
<div style='border-left:6px solid #C0392B;padding-left:20px'>
<h1 style='color:white;margin:0;font-size:32px'>🔪 Hunting the Pattern</h1>
<h2 style='color:#AAAAAA;margin:8px 0 0;font-weight:400;font-size:18px'>Geographic Profiling of Serial Killers — Complete Analysis</h2>
</div>
</div>

---

> *"Serial killers don't operate in a vacuum — they operate in the blind spots of the system.  
> Data shows where those blind spots are, when they appeared, and why some victims are invisible from the start."*

---

### Project structure

| Module | Focus | Key output |
|--------|-------|------------|
| **01** | EDA & Offender Profiling | Temporal trends · geographic distribution · victim counts · gender & method |
| **02** | Geographic Analysis | Continent heatmaps · per-capita ranking · interactive Folium maps |
| **03** | Predictive Modelling | Random Forest · feature importance · partial effects · model comparison |

### Data sources
- **Kaggle** — Serial Killers by Victim Count (305 killers)
- **Wikipedia** — List of Serial Killers by Country (328 killers, parsed from PDF)
- **Master dataset:** 633 killers · 101 countries · 14 features

### Key finding
Victim counts follow a power law. Geography predicts lethality. Era predicts time to capture.  
Most importantly: **victims become socially invisible before the killing even starts.**

### Stack
`Python` · `pandas` · `matplotlib` · `seaborn` · `scikit-learn` · `Folium`

**Author:** [Your Name] | **Year:** 2025

---

In [ ]:
# Install dependencies (run once if needed)
# !pip install pandas numpy matplotlib seaborn scikit-learn folium

# ── Place serial_killers_master.csv in the same folder as this notebook ──
print('✓ Ready. Run all cells from top to bottom (Kernel → Restart & Run All).')

---

<div style='background:#1A1A2E;padding:20px 24px;border-radius:8px;border-left:6px solid #C0392B;margin:20px 0'>
<span style='color:#C0392B;font-size:13px;font-weight:600;letter-spacing:2px'>MODULE 01</span><br>
<span style='color:white;font-size:22px;font-weight:700'>EDA & Offender Profiling</span><br>
<span style='color:#AAAAAA;font-size:13px'>Temporal trends · geographic distribution · victim counts · gender & method</span>
</div>

---


## 0. Setup

In [ ]:
# Uncomment to install on first run
# !pip install pandas numpy matplotlib seaborn

In [ ]:
pip install numpy==1.26.4 --force-reinstall

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from matplotlib.gridspec import GridSpec

# ── McKinsey-inspired visual style ────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor':  'white',
    'axes.facecolor':    'white',
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'axes.spines.left':  False,
    'axes.grid':         True,
    'grid.color':        '#EEEEEE',
    'grid.linewidth':    0.6,
    'axes.axisbelow':    True,
    'font.family':       'sans-serif',
    'font.size':         11,
    'axes.titlesize':    14,
    'axes.titleweight':  'bold',
    'axes.labelsize':    11,
    'xtick.labelsize':   10,
    'ytick.labelsize':   10,
    'legend.frameon':    False,
    'figure.dpi':        120,
    'savefig.dpi':       150,
    'savefig.bbox':      'tight',
})

# Brand palette
C = {
    'primary':   '#1A1A2E',
    'red':       '#C0392B',
    'red_light': '#E74C3C',
    'blue':      '#2980B9',
    'gray':      '#7F8C8D',
    'light':     '#ECF0F1',
    'green':     '#27AE60',
    'orange':    '#E67E22',
}

print('Setup complete.')

---
## 1. Load Data

Place `serial_killers_master.csv` in the same folder as this notebook.

In [ ]:
# ── Load ──────────────────────────────────────────────────────────────────────
df = pd.read_csv('serial_killers_master.csv')

# Numeric coercion
for col in ['victims_proven', 'victims_possible', 'victims_max',
            'year_start', 'year_end', 'years_span', 'decade']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Modern era subset (pre-1900 is very sparse and methodologically different)
df_modern = df[df['year_start'] >= 1900].copy()

print(f'Total killers:      {len(df):,}')
print(f'Countries:          {df["country"].nunique()}')
print(f'Year range:         {int(df["year_start"].min())} – {int(df["year_start"].max())}')
print(f'With victims:       {df["victims_proven"].notna().sum()}')
print(f'With method:        {df["method"].notna().sum()}')
print(f'With gender:        {df["gender"].notna().sum()}')
print(f'\nData sources:')
print(df['source'].value_counts().to_string())

In [ ]:
df[['name', 'country', 'victims_proven', 'year_start', 'year_end',
    'method', 'status', 'gender']].head(8)

---
## 2. Temporal Analysis — When Did Serial Killers Peak?

> **Hypothesis:** The 1980–1990s saw a global surge in documented serial killings,  
> driven partly by better forensics, media coverage, and criminal databases — not just more killers.

In [ ]:
df_dec = df_modern.dropna(subset=['decade']).copy()
df_dec['decade']  = df_dec['decade'].astype(int)
df_dec['us_flag'] = df_dec['country'].str.upper() == 'UNITED STATES'

decade_pivot = (
    df_dec.groupby(['decade', 'us_flag'])
    .size()
    .unstack(fill_value=0)
    .rename(columns={True: 'United States', False: 'International'})
    .loc[lambda d: (d.index >= 1900) & (d.index <= 2010)]
)

fig, ax = plt.subplots(figsize=(13, 5))
x = np.arange(len(decade_pivot))
w = 0.42

ax.bar(x - w/2, decade_pivot['United States'],  width=w, color=C['red'],  label='United States', zorder=3)
ax.bar(x + w/2, decade_pivot['International'], width=w, color=C['blue'], label='International', alpha=0.85, zorder=3)

ax.set_xticks(x)
ax.set_xticklabels([f"{d}s" for d in decade_pivot.index], rotation=45, ha='right')
ax.set_ylabel('Killers (by decade of first kill)')
ax.set_title('Serial Killer Activity by Decade — US vs. International', pad=14)
ax.legend()

# Annotate peaks
for col, offset, ec in [('United States', -w/2, C['red']), ('International', +w/2, C['blue'])]:
    peak_dec = decade_pivot[col].idxmax()
    peak_val = decade_pivot.loc[peak_dec, col]
    peak_x   = list(decade_pivot.index).index(peak_dec)
    label    = f"{col}\npeak: {peak_dec}s ({peak_val})"
    ax.annotate(label,
        xy=(peak_x + offset, peak_val),
        xytext=(peak_x + offset + (1.5 if offset > 0 else -1.5), peak_val + 4),
        arrowprops=dict(arrowstyle='->', color=C['primary'], lw=1.1),
        fontsize=8.5, color=C['primary'],
        bbox=dict(boxstyle='round,pad=0.3', fc='white', ec=ec, lw=0.8))

plt.tight_layout()
plt.savefig('01_decade_trend.png')
plt.show()

us_peak = decade_pivot['United States'].idxmax()
gl_peak = decade_pivot['International'].idxmax()
print(f'US peak decade:     {us_peak}s')
print(f'Global peak decade: {gl_peak}s')

---
## 3. Geographic Analysis — Which Countries Lead?

In [ ]:
# ── Top 20 countries by raw count ─────────────────────────────────────────────
country_counts = df['country'].str.strip().str.title().value_counts().head(20)

color_list = [C['red']] + [C['blue']] * 4 + [C['gray']] * (len(country_counts) - 5)

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(country_counts.index[::-1], country_counts.values[::-1],
               color=color_list[::-1], zorder=3)
ax.set_xlabel('Documented serial killers')
ax.set_title('Top 20 Countries by Serial Killer Count', pad=14)
ax.bar_label(bars, padding=4, fontsize=9, color=C['gray'])
ax.set_xlim(0, country_counts.max() * 1.15)

us_pct = country_counts.get('United States', 0) / len(df)
ax.text(0.98, 0.02, f'US = {us_pct:.0%} of total dataset',
        transform=ax.transAxes, ha='right', va='bottom',
        fontsize=9, color=C['gray'], style='italic')

legend_items = [mpatches.Patch(color=C['red'], label='#1'),
                mpatches.Patch(color=C['blue'], label='Top 2–5'),
                mpatches.Patch(color=C['gray'], label='Rest')]
ax.legend(handles=legend_items, loc='lower right')

plt.tight_layout()
plt.savefig('02_top_countries.png')
plt.show()

print(f'Countries with documented killers: {df["country"].nunique()}')
print(f'Top 5 share: {country_counts.head(5).sum()/len(df):.0%}')

In [ ]:
# ── Per-capita adjustment ─────────────────────────────────────────────────────
# Approximate 2000s population (millions)
pop = {
    'United States': 300, 'Russia': 145, 'Italy': 58,
    'Australia': 20,  'India': 1100, 'Canada': 33,
    'South Africa': 47, 'Poland': 38, 'South Korea': 48,
    'Ukraine': 46,  'United Kingdom': 61, 'Belgium': 11,
    'Iran': 70, 'Mexico': 105, 'Argentina': 40,
    'Germany': 82,  'France': 63,  'Japan': 128,
}

pc = []
for c, n in country_counts.items():
    if c in pop:
        pc.append({'country': c, 'count': n, 'per_10m': round(n / pop[c] * 10, 2)})

df_cap = pd.DataFrame(pc).sort_values('per_10m', ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
colors = [C['red'] if r['country'] == 'United States' else
          C['blue'] if r['per_10m'] > df_cap['per_10m'].median() else C['gray']
          for _, r in df_cap.iterrows()]

bars = ax.barh(df_cap['country'][::-1], df_cap['per_10m'][::-1],
               color=colors[::-1], zorder=3)
ax.set_xlabel('Serial killers per 10 million population (approx.)')
ax.set_title('Per-Capita View — Does the US Still Lead After Adjusting for Size?', pad=14)
ax.bar_label(bars, fmt='%.1f', padding=3, fontsize=9, color=C['gray'])
ax.text(0.98, 0.02,
        'Note: pop. estimates approx. (2000s)\nDocumentation bias affects smaller countries',
        transform=ax.transAxes, ha='right', va='bottom',
        fontsize=8, color=C['gray'], style='italic')

plt.tight_layout()
plt.savefig('03_per_capita.png')
plt.show()

---
## 4. Victim Count Distribution — The Long Tail of Evil

In [ ]:
vic = df['victims_proven'].dropna()
vic_trim = vic[vic <= 150]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Distribution of Proven Victim Counts', fontsize=15, fontweight='bold')

# Raw
ax = axes[0]
ax.hist(vic_trim, bins=35, color=C['red'], edgecolor='white', linewidth=0.4, zorder=3)
ax.axvline(vic.median(), color=C['primary'], lw=1.8, ls='--', label=f'Median: {vic.median():.0f}')
ax.axvline(vic.mean(),   color=C['blue'],    lw=1.8, ls=':',  label=f'Mean: {vic.mean():.1f}')
ax.set_xlabel('Proven victims')
ax.set_ylabel('Number of killers')
ax.set_title('Raw distribution (capped at 150)')
ax.legend()

# Log
ax = axes[1]
ax.hist(np.log1p(vic), bins=35, color=C['blue'], edgecolor='white', linewidth=0.4, zorder=3)
ax.set_xlabel('log(proven victims + 1)')
ax.set_title('Log scale — long tail becomes visible')

top10 = vic[vic >= vic.quantile(0.9)]
share = top10.sum() / vic.sum()
ax.text(0.97, 0.94,
        f'Most killers: 3–6 victims\nTop 10% account for\n~{share:.0%} of all deaths',
        transform=ax.transAxes, ha='right', va='top', fontsize=9, color=C['primary'],
        bbox=dict(boxstyle='round,pad=0.4', fc='#FEF9E7', ec=C['gray'], lw=0.8))

plt.tight_layout()
plt.savefig('04_victim_distribution.png')
plt.show()

print('── Victim count statistics ──────────────────────────────────')
for k, v in vic.describe().round(1).items():
    print(f'  {k:<10}: {v}')
print(f'  90th pct : {vic.quantile(0.9):.0f}')
print(f'  Top 10% ({len(top10)} killers) → {share:.0%} of all proven deaths')

---
## 5. Offender Profile — Gender, Method, Fate

In [ ]:
fig = plt.figure(figsize=(15, 5))
fig.suptitle('Offender Profile Overview', fontsize=15, fontweight='bold', y=1.01)
gs = GridSpec(1, 3, figure=fig, wspace=0.38)

# ── Gender pie ────────────────────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0])
gc = df['gender'].dropna().str.strip().str.title().value_counts()
wedges, texts, autotexts = ax1.pie(
    gc.values, labels=gc.index, autopct='%1.1f%%',
    colors=[C['blue'], C['red'], C['gray']][:len(gc)],
    startangle=90, wedgeprops=dict(edgecolor='white', linewidth=2.5),
    textprops={'fontsize': 10}
)
for at in autotexts:
    at.set_fontsize(9); at.set_color('white'); at.set_fontweight('bold')
ax1.set_title('Gender of killer', pad=12)

# ── Method bar ────────────────────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[1])
mc = df['method'].dropna().str.strip().value_counts()
mc_colors = [C['red']] + [C['blue']] * 3 + [C['gray']] * max(0, len(mc) - 4)
bars = ax2.barh(mc.index[::-1], mc.values[::-1], color=mc_colors[::-1], zorder=3)
ax2.set_xlabel('Count')
ax2.set_title('Method of killing', pad=12)
ax2.bar_label(bars, padding=3, fontsize=9, color=C['gray'])
ax2.set_xlim(0, mc.max() * 1.2)

# ── Status bar ────────────────────────────────────────────────────────────────
ax3 = fig.add_subplot(gs[2])
sc = df['status'].dropna().str.strip().value_counts()
sc_colors = [C['red'], C['blue'], C['orange'], C['gray'], C['green']][:len(sc)]
bars3 = ax3.bar(range(len(sc)), sc.values, color=sc_colors, zorder=3, width=0.6)
ax3.set_xticks(range(len(sc)))
ax3.set_xticklabels([s.replace(' ', '\n') for s in sc.index], fontsize=9)
ax3.set_ylabel('Count')
ax3.set_title('Outcome / fate', pad=12)
ax3.bar_label(bars3, padding=3, fontsize=9, color=C['gray'])

plt.savefig('05_offender_profile.png')
plt.show()

print(f'Male share:    {gc.get("Male", 0)/gc.sum():.0%}')
print(f'Top method:    {mc.index[0]} ({mc.iloc[0]} cases)')
print(f'Executed:      {sc.get("Executed", 0)/sc.sum():.0%} of known outcomes')

---
## 6. Method × Decade Heatmap — How Killing Changed Over Time

In [ ]:
df_heat = df_modern.dropna(subset=['method', 'decade']).copy()
df_heat['decade'] = df_heat['decade'].astype(int)
df_heat = df_heat[(df_heat['decade'] >= 1940) & (df_heat['decade'] <= 2010)]

heat_pivot = (
    df_heat.groupby(['decade', 'method'])
    .size()
    .unstack(fill_value=0)
)

if not heat_pivot.empty:
    fig, ax = plt.subplots(figsize=(12, 5))
    sns.heatmap(
        heat_pivot.T, ax=ax, cmap='Reds',
        linewidths=0.8, linecolor='white',
        annot=True, fmt='d', annot_kws={'size': 9},
        cbar_kws={'label': 'Number of killers'}
    )
    ax.set_xlabel('Decade', labelpad=8)
    ax.set_ylabel('')
    ax.set_title('Killing Method by Decade (1940–2010)', pad=14)
    ax.set_xticklabels([f"{x.get_text()}s" for x in ax.get_xticklabels()], rotation=0)
    plt.tight_layout()
    plt.savefig('06_method_heatmap.png')
    plt.show()
else:
    print('Not enough method × decade data.')

---
## 7. Victim Count by Country — Where Is Lethality Highest?

In [ ]:
top_countries = df['country'].value_counts().head(12).index

country_stats = (
    df[df['country'].isin(top_countries)]
    .groupby('country')['victims_proven']
    .agg(['mean', 'median', 'sum', 'count'])
    .round(1)
    .sort_values('mean', ascending=False)
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Victim Counts by Country (Top 12 by frequency)', fontsize=14, fontweight='bold')

for ax, col, title, fmt in [
    (axes[0], 'mean', 'Average victims per killer', '%.1f'),
    (axes[1], 'sum',  'Total victims (proven)',     '%.0f')
]:
    sorted_stats = country_stats.sort_values(col, ascending=False)
    colors = [C['red'] if c == 'United States' else
              C['blue'] if sorted_stats[col][c] > sorted_stats[col].median() else C['gray']
              for c in sorted_stats.index]
    bars = ax.barh(sorted_stats.index[::-1], sorted_stats[col][::-1],
                   color=colors[::-1], zorder=3)
    ax.set_xlabel(title)
    ax.set_title(title)
    ax.bar_label(bars, fmt=fmt, padding=3, fontsize=9, color=C['gray'])

plt.tight_layout()
plt.savefig('07_victims_by_country.png')
plt.show()

print('── Country victim stats (killers with known victim count only) ──')
print(country_stats.to_string())

---
## 8. Active Duration — How Long Before Capture?

In [ ]:
df_span = df.dropna(subset=['years_span']).copy()
df_span = df_span[(df_span['years_span'] >= 0) & (df_span['years_span'] <= 40)]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('How Long Did They Operate Before Being Caught?', fontsize=14, fontweight='bold')

# Distribution
ax = axes[0]
ax.hist(df_span['years_span'], bins=30, color=C['red'],
        edgecolor='white', linewidth=0.4, zorder=3)
ax.axvline(df_span['years_span'].median(), color=C['primary'], lw=1.8, ls='--',
           label=f"Median: {df_span['years_span'].median():.0f} yr(s)")
ax.set_xlabel('Years active')
ax.set_ylabel('Number of killers')
ax.set_title('Distribution of active years')
ax.legend()

# Scatter: years active vs victim count
ax = axes[1]
df_sc = df_span.dropna(subset=['victims_proven'])
df_sc = df_sc[df_sc['victims_proven'] <= 100]

ax.scatter(df_sc['years_span'], df_sc['victims_proven'],
           alpha=0.4, s=30, color=C['blue'],
           edgecolors=C['primary'], linewidths=0.3)

if len(df_sc) > 10:
    z = np.polyfit(df_sc['years_span'], df_sc['victims_proven'], 1)
    x_line = np.linspace(0, 40, 100)
    ax.plot(x_line, np.poly1d(z)(x_line), color=C['red'], lw=1.5, ls='--', label='Trend')
    corr = df_sc[['years_span', 'victims_proven']].corr().iloc[0, 1]
    ax.text(0.97, 0.95, f'Pearson r = {corr:.2f}',
            transform=ax.transAxes, ha='right', va='top', fontsize=9, color=C['primary'],
            bbox=dict(boxstyle='round,pad=0.3', fc='white', ec=C['gray'], lw=0.8))
    ax.legend()

ax.set_xlabel('Years active')
ax.set_ylabel('Proven victims')
ax.set_title('Longer active = more victims?')

plt.tight_layout()
plt.savefig('08_active_duration.png')
plt.show()

print(f'Median active years:    {df_span["years_span"].median():.0f}')
print(f'Single-year killers:    {(df_span["years_span"] == 0).sum()} ({(df_span["years_span"]==0).mean():.0%})')
print(f'Active 10+ years:       {(df_span["years_span"] >= 10).sum()} ({(df_span["years_span"]>=10).mean():.0%})')

---
## 9. Gender Deep-Dive — Do Female Killers Differ?

In [ ]:
df_g = df.dropna(subset=['gender']).copy()
df_g['gender'] = df_g['gender'].str.strip().str.title()

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Male vs. Female Serial Killers — Key Differences', fontsize=14, fontweight='bold')

# Method preference
ax = axes[0]
mg = (df_g.dropna(subset=['method'])
      .groupby(['gender', 'method']).size()
      .unstack(fill_value=0)
      .apply(lambda x: x / x.sum(), axis=1))
if not mg.empty:
    mg.T.plot(kind='bar', ax=ax, color=[C['blue'], C['red']][:len(mg)],
              width=0.7, zorder=3)
    ax.set_ylabel('Share of killers')
    ax.set_title('Method preference')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
    ax.legend(title='Gender')

# Victim count boxplot
ax = axes[1]
df_vg = df_g.dropna(subset=['victims_proven'])
df_vg = df_vg[df_vg['victims_proven'] <= 80]
groups = [df_vg[df_vg['gender'] == g]['victims_proven'].values
          for g in ['Male', 'Female'] if g in df_vg['gender'].values]
labels  = [g for g in ['Male', 'Female'] if g in df_vg['gender'].values]
if groups:
    bp = ax.boxplot(groups, labels=labels, patch_artist=True,
                    medianprops=dict(color='white', lw=2),
                    whiskerprops=dict(color=C['gray']),
                    capprops=dict(color=C['gray']),
                    flierprops=dict(marker='o', markerfacecolor=C['red'],
                                    markersize=3, alpha=0.5, linestyle='none'))
    for patch, col in zip(bp['boxes'], [C['blue'], C['red']]):
        patch.set_facecolor(col); patch.set_alpha(0.8)
ax.set_ylabel('Proven victims')
ax.set_title('Victim count by gender')

# Outcome
ax = axes[2]
sg = (df_g.dropna(subset=['status'])
      .groupby(['gender', 'status']).size()
      .unstack(fill_value=0)
      .apply(lambda x: x / x.sum(), axis=1))
if not sg.empty:
    sg.T.plot(kind='bar', ax=ax, color=[C['blue'], C['red']][:len(sg)],
              width=0.7, zorder=3)
    ax.set_ylabel('Share')
    ax.set_title('Outcome by gender')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
    ax.legend(title='Gender')

plt.tight_layout()
plt.savefig('09_gender_analysis.png')
plt.show()

---
## 10. Key Findings Summary

In [ ]:
# ── Auto-compute all key stats ────────────────────────────────────────────────
us_n       = (df['country'].str.upper() == 'UNITED STATES').sum()
top_dec    = df_modern.dropna(subset=['decade'])['decade'].astype(int).value_counts().idxmax()
med_vic    = df['victims_proven'].dropna().median()
med_span   = df.dropna(subset=['years_span'])['years_span'].median()
top_meth   = df['method'].dropna().value_counts().idxmax()
male_pct   = (df['gender'].dropna().str.strip().str.title() == 'Male').mean()
top5_pct   = df['country'].value_counts().head(5).sum() / len(df)
vic_top10  = df['victims_proven'].dropna()
vic_top10  = vic_top10[vic_top10 >= vic_top10.quantile(0.9)].sum() / vic_top10.sum()

print('═' * 62)
print('  HUNTING THE PATTERN — KEY FINDINGS')
print('  Serial Killers: Global Data Science Analysis')
print('═' * 62)
print(f'\n  DATASET')
print(f'  • {len(df):,} killers documented across {df["country"].nunique()} countries')
print(f'  • Sources: Kaggle + Wikipedia (by country), merged & deduplicated')
print(f'\n  GEOGRAPHY')
print(f'  • US: {us_n} killers ({us_n/len(df):.0%} of global total)')
print(f'  • Top 5 countries = {top5_pct:.0%} of all documented cases')
print(f'  • 101 countries have at least one documented case')
print(f'\n  TEMPORAL')
print(f'  • Peak decade: {top_dec}s')
print(f'  • Activity rose sharply after 1960, peaked ~{top_dec}s, declining since')
print(f'\n  VICTIMS')
print(f'  • Median proven victims: {med_vic:.0f} per killer')
print(f'  • Top 10% of killers account for ~{vic_top10:.0%} of all deaths')
print(f'  • Most common documented method: {top_meth}')
print(f'\n  PROFILE')
print(f'  • {male_pct:.0%} male (among killers with known gender)')
print(f'  • Median active period before capture: {med_span:.0f} year(s)')
print(f'  • Female killers strongly favor poisoning; male killers more varied')
print(f'\n  ── NEXT: Module 02 → Geographic clustering & hotspot maps')
print('═' * 62)

---

## What's Next

| Module | Focus | Techniques |
|--------|-------|------------|
| **02** | Geographic clustering | Folium choropleth, DBSCAN hotspots |
| **03** | Predictive modelling | Random Forest, XGBoost, SHAP |
| **04** | Storytelling output | LinkedIn carousel, portfolio slides |

---
*Data: Kaggle Serial Killers Dataset + Wikipedia List of Serial Killers by Country*  
*Combined, cleaned & analysed by [Natalia Prokofyeva], 2026*

---

<div style='background:#1A1A2E;padding:20px 24px;border-radius:8px;border-left:6px solid #C0392B;margin:20px 0'>
<span style='color:#C0392B;font-size:13px;font-weight:600;letter-spacing:2px'>MODULE 02</span><br>
<span style='color:white;font-size:22px;font-weight:700'>Geographic Analysis</span><br>
<span style='color:#AAAAAA;font-size:13px'>Continent heatmaps · per-capita ranking · interactive Folium maps · decade breakdown</span>
</div>

---


## 0. Setup

In [ ]:
# Install if needed
# !pip install folium

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import seaborn as sns
import folium
from folium.plugins import HeatMap
import json
import os

# ── Visual style ──────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor':  'white', 'axes.facecolor':    'white',
    'axes.spines.top':   False,   'axes.spines.right': False,
    'axes.spines.left':  False,   'axes.grid':         True,
    'grid.color':        '#EEEEEE','grid.linewidth':    0.6,
    'axes.axisbelow':    True,    'font.family':       'sans-serif',
    'font.size':         11,      'axes.titlesize':    14,
    'axes.titleweight':  'bold',  'legend.frameon':    False,
    'figure.dpi':        120,     'savefig.dpi':       150,
    'savefig.bbox':      'tight',
})

C = {
    'primary': '#1A1A2E', 'red':    '#C0392B', 'red_light': '#E74C3C',
    'blue':    '#2980B9', 'gray':   '#7F8C8D', 'green':     '#27AE60',
    'orange':  '#E67E22', 'purple': '#8E44AD', 'teal':      '#16A085',
}

print('Setup complete.')

---
## 1. Load & Prepare Data

In [ ]:
df = pd.read_csv('serial_killers_master.csv')

for col in ['victims_proven', 'victims_possible', 'victims_max',
            'year_start', 'year_end', 'years_span', 'decade']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# ── Country-level aggregation ─────────────────────────────────────────────────
country_agg = (
    df.groupby('country')
    .agg(
        killers       = ('name',          'count'),
        victims_total = ('victims_proven', 'sum'),
        victims_avg   = ('victims_proven', 'mean'),
        victims_max   = ('victims_proven', 'max'),
        year_min      = ('year_start',    'min'),
        year_max      = ('year_start',    'max'),
    )
    .round(1)
    .reset_index()
)

# ── Population data for per-capita ───────────────────────────────────────────
pop = {
    'United States': 300, 'Russia': 145, 'Italy': 58, 'Australia': 20,
    'India': 1100, 'Canada': 33, 'South Africa': 47, 'Poland': 38,
    'South Korea': 48, 'Ukraine': 46, 'United Kingdom': 61, 'Belgium': 11,
    'Iran': 70, 'Mexico': 105, 'Argentina': 40, 'Germany': 82,
    'France': 63, 'Japan': 128, 'Spain': 40, 'China': 1300,
    'Brazil': 190, 'Austria': 8, 'Indonesia': 220, 'Finland': 5,
    'Turkey': 70, 'Netherlands': 16, 'Greece': 11, 'Colombia': 44,
    'Belarus': 10, 'Thailand': 65, 'Hungary': 10, 'Romania': 21,
    'Kazakhstan': 15, 'Sweden': 9, 'Norway': 5, 'Denmark': 5,
    'Czech Republic': 10, 'Slovakia': 5, 'Latvia': 2, 'Estonia': 1,
    'Lithuania': 3, 'Serbia': 7, 'Croatia': 4, 'Slovenia': 2,
    'Malaysia': 26, 'Philippines': 88, 'Pakistan': 165, 'Bangladesh': 150,
    'Nigeria': 150, 'Egypt': 75, 'Morocco': 31, 'Zimbabwe': 12,
    'Zambia': 12, 'New Zealand': 4, 'Singapore': 4, 'Taiwan': 23,
    'Iceland': 0.3, 'Malta': 0.4, 'Cyprus': 0.8, 'Luxembourg': 0.5,
}
country_agg['population_m'] = country_agg['country'].map(pop)
country_agg['killers_per_10m'] = (
    country_agg['killers'] / country_agg['population_m'] * 10
).round(2)

print(f'Countries in dataset: {len(country_agg)}')
print(f'Countries with population data: {country_agg["population_m"].notna().sum()}')
print(country_agg.sort_values('killers', ascending=False).head(10).to_string(index=False))

---
## 2. Static Geographic Charts

> These charts work without Folium and are ready for LinkedIn slides.

In [ ]:
# ── 2a. Continent mapping ─────────────────────────────────────────────────────
continent_map = {
    'United States': 'North America', 'Canada': 'North America',
    'Mexico': 'North America', 'Colombia': 'South America',
    'Brazil': 'South America', 'Argentina': 'South America',
    'Peru': 'South America', 'Venezuela': 'South America',
    'Russia': 'Europe', 'Italy': 'Europe', 'Poland': 'Europe',
    'Ukraine': 'Europe', 'United Kingdom': 'Europe', 'Belgium': 'Europe',
    'Germany': 'Europe', 'France': 'Europe', 'Spain': 'Europe',
    'Austria': 'Europe', 'Netherlands': 'Europe', 'Greece': 'Europe',
    'Hungary': 'Europe', 'Romania': 'Europe', 'Finland': 'Europe',
    'Sweden': 'Europe', 'Norway': 'Europe', 'Denmark': 'Europe',
    'Czech Republic': 'Europe', 'Slovakia': 'Europe', 'Latvia': 'Europe',
    'Estonia': 'Europe', 'Lithuania': 'Europe', 'Belarus': 'Europe',
    'Serbia': 'Europe', 'Croatia': 'Europe', 'Slovenia': 'Europe',
    'Portugal': 'Europe', 'Switzerland': 'Europe', 'Bulgaria': 'Europe',
    'North Macedonia': 'Europe', 'Malta': 'Europe', 'Cyprus': 'Europe',
    'Iceland': 'Europe', 'Ireland': 'Europe', 'Luxembourg': 'Europe',
    'Moldova': 'Europe', 'Kazakhstan': 'Asia', 'Iran': 'Asia',
    'India': 'Asia', 'China': 'Asia', 'South Korea': 'Asia',
    'Japan': 'Asia', 'Indonesia': 'Asia', 'Thailand': 'Asia',
    'Malaysia': 'Asia', 'Philippines': 'Asia', 'Pakistan': 'Asia',
    'Bangladesh': 'Asia', 'Taiwan': 'Asia', 'Singapore': 'Asia',
    'Lebanon': 'Asia', 'Israel': 'Asia', 'Iraq': 'Asia',
    'Jordan': 'Asia', 'Saudi Arabia': 'Asia', 'Yemen': 'Asia',
    'Turkey': 'Asia', 'Georgia': 'Asia', 'Kyrgyzstan': 'Asia',
    'Uzbekistan': 'Asia', 'Vietnam': 'Asia',
    'South Africa': 'Africa', 'Nigeria': 'Africa', 'Egypt': 'Africa',
    'Morocco': 'Africa', 'Zimbabwe': 'Africa', 'Zambia': 'Africa',
    'Rwanda': 'Africa', 'Eswatini': 'Africa', 'Ghana': 'Africa',
    'Algeria': 'Africa', 'Tunisia': 'Africa', 'Namibia': 'Africa',
    'Belgian Congo': 'Africa',
    'Australia': 'Oceania', 'New Zealand': 'Oceania', 'Fiji': 'Oceania',
    'Afghanistan': 'Asia',
}

df['continent'] = df['country'].map(continent_map).fillna('Other')
country_agg['continent'] = country_agg['country'].map(continent_map).fillna('Other')

continent_stats = (
    df.groupby('continent')
    .agg(
        killers       = ('name',          'count'),
        victims_total = ('victims_proven', 'sum'),
        victims_avg   = ('victims_proven', 'mean'),
    )
    .round(1)
    .sort_values('killers', ascending=False)
)

print(continent_stats.to_string())

In [ ]:
# ── 2b. Continent overview — killers + lethality ──────────────────────────────
cont = continent_stats[continent_stats.index != 'Other'].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Serial Killers by Continent', fontsize=15, fontweight='bold')

cont_colors = [C['red'], C['blue'], C['teal'], C['orange'],
               C['purple'], C['green'], C['gray']]

# Killers count
ax = axes[0]
sorted_k = cont.sort_values('killers', ascending=True)
bars = ax.barh(sorted_k.index, sorted_k['killers'],
               color=cont_colors[:len(sorted_k)][::-1], zorder=3)
ax.set_xlabel('Number of documented killers')
ax.set_title('Killers by continent')
ax.bar_label(bars, padding=4, fontsize=10, color=C['gray'])
ax.set_xlim(0, sorted_k['killers'].max() * 1.18)

# Average victims per killer
ax = axes[1]
sorted_v = cont.sort_values('victims_avg', ascending=True).dropna(subset=['victims_avg'])
bars2 = ax.barh(sorted_v.index, sorted_v['victims_avg'],
                color=cont_colors[:len(sorted_v)][::-1], zorder=3)
ax.set_xlabel('Average proven victims per killer')
ax.set_title('Average lethality by continent')
ax.bar_label(bars2, fmt='%.1f', padding=4, fontsize=10, color=C['gray'])
ax.set_xlim(0, sorted_v['victims_avg'].max() * 1.18)

plt.tight_layout()
plt.savefig('02a_continent_overview.png')
plt.show()

print('\n── Continent insight ─────────────────────────────────────────')
top_lethal = cont['victims_avg'].idxmax()
print(f'Most killers:     {cont["killers"].idxmax()} ({cont["killers"].max():.0f})')
print(f'Most lethal avg:  {top_lethal} ({cont.loc[top_lethal,"victims_avg"]:.1f} avg victims)')

In [ ]:
# ── 2c. Dual-axis: killers count vs average victims (bubble) ──────────────────
top20 = country_agg.sort_values('killers', ascending=False).head(20).copy()
top20['victims_avg'] = top20['victims_avg'].fillna(top20['victims_avg'].median())

fig, ax = plt.subplots(figsize=(12, 7))

scatter = ax.scatter(
    top20['killers'],
    top20['victims_avg'],
    s=top20['killers'] * 8,
    c=top20['victims_total'].fillna(0),
    cmap='Reds', alpha=0.75,
    edgecolors=C['primary'], linewidths=0.5, zorder=3
)

plt.colorbar(scatter, ax=ax, label='Total proven victims', shrink=0.7)

for _, row in top20.iterrows():
    ax.annotate(
        row['country'],
        (row['killers'], row['victims_avg']),
        textcoords='offset points', xytext=(6, 3),
        fontsize=8.5, color=C['primary']
    )

ax.set_xlabel('Number of documented killers')
ax.set_ylabel('Average proven victims per killer')
ax.set_title(
    'Volume vs Lethality — Top 20 Countries\n'
    '(bubble size = number of killers, colour = total victims)',
    pad=12
)

ax.axhline(top20['victims_avg'].median(), color=C['gray'], lw=1, ls='--', alpha=0.6,
           label=f'Median avg victims: {top20["victims_avg"].median():.1f}')
ax.axvline(top20['killers'].median(), color=C['gray'], lw=1, ls=':', alpha=0.6,
           label=f'Median killers: {top20["killers"].median():.0f}')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('02b_volume_vs_lethality.png')
plt.show()

In [ ]:
# ── 2d. Per-capita ranking ────────────────────────────────────────────────────
df_cap = country_agg.dropna(subset=['killers_per_10m']).sort_values(
    'killers_per_10m', ascending=False
).head(18)

fig, ax = plt.subplots(figsize=(10, 7))

bar_colors = [
    C['red']  if r['country'] == 'United States' else
    C['blue'] if r['killers_per_10m'] > df_cap['killers_per_10m'].median() else
    C['gray']
    for _, r in df_cap.iterrows()
]

bars = ax.barh(df_cap['country'][::-1], df_cap['killers_per_10m'][::-1],
               color=bar_colors[::-1], zorder=3)
ax.bar_label(bars, fmt='%.1f', padding=3, fontsize=9, color=C['gray'])
ax.set_xlabel('Serial killers per 10 million population')
ax.set_title(
    'Per-Capita Ranking — Who Actually Leads After Adjusting for Population?',
    pad=12
)

ax.text(0.98, 0.02,
        'Population: approximate 2000s estimates.\n'
        'Documentation bias likely affects smaller countries.',
        transform=ax.transAxes, ha='right', va='bottom',
        fontsize=8, color=C['gray'], style='italic')

plt.tight_layout()
plt.savefig('02c_per_capita_ranking.png')
plt.show()

In [ ]:
# ── 2e. Decade × Continent heatmap ────────────────────────────────────────────
df_heat = df[df['year_start'] >= 1900].dropna(subset=['decade']).copy()
df_heat['decade'] = df_heat['decade'].astype(int)
df_heat = df_heat[(df_heat['decade'] >= 1920) & (df_heat['decade'] <= 2010)]
df_heat = df_heat[df_heat['continent'] != 'Other']

heat_pivot = (
    df_heat.groupby(['decade', 'continent'])
    .size()
    .unstack(fill_value=0)
)

fig, ax = plt.subplots(figsize=(13, 5))
sns.heatmap(
    heat_pivot.T,
    ax=ax, cmap='Reds',
    linewidths=0.8, linecolor='white',
    annot=True, fmt='d', annot_kws={'size': 9},
    cbar_kws={'label': 'Killers (by decade of first kill)'}
)
ax.set_xlabel('Decade', labelpad=8)
ax.set_ylabel('')
ax.set_title('Serial Killer Activity: Continent × Decade (1920–2010)', pad=14)
ax.set_xticklabels([f"{x.get_text()}s" for x in ax.get_xticklabels()], rotation=0)

plt.tight_layout()
plt.savefig('02d_continent_decade_heatmap.png')
plt.show()

---
## 3. Interactive Folium Map — World Choropleth

> Saves as `serial_killers_map.html` — embed on your portfolio site as an iframe.

In [ ]:
# ── Download world GeoJSON (run once) ─────────────────────────────────────────
import urllib.request

geojson_url  = 'https://raw.githubusercontent.com/datasets/geo-countries/master/data/countries.geojson'
geojson_path = 'world_countries.geojson'

if not os.path.exists(geojson_path):
    print('Downloading world GeoJSON...')
    urllib.request.urlretrieve(geojson_url, geojson_path)
    print('Done.')
else:
    print('GeoJSON already downloaded.')

In [ ]:
# ── Country name harmonisation (dataset → GeoJSON names) ─────────────────────
# GeoJSON uses ISO country names — align ours where they differ
name_fix = {
    'United States':    'United States of America',
    'United Kingdom':   'United Kingdom',
    'South Korea':      'Republic of Korea',
    'North Korea':      "Democratic People's Republic of Korea",
    'Russia':           'Russia',
    'Iran':             'Iran (Islamic Republic of)',
    'Czech Republic':   'Czechia',
    'Serbia':           'Serbia',
    'North Macedonia':  'North Macedonia',
    'Moldova':          'Republic of Moldova',
    'Taiwan':           'Taiwan',
    'South Africa':     'South Africa',
    'Belgian Congo':    'Democratic Republic of the Congo',
    'Eswatini':         'eSwatini',
}

country_agg['country_geo'] = country_agg['country'].replace(name_fix)

# Prep lookup dicts
killers_dict = dict(zip(country_agg['country_geo'], country_agg['killers']))
victims_dict = dict(zip(country_agg['country_geo'], country_agg['victims_total'].fillna(0)))
avg_dict     = dict(zip(country_agg['country_geo'], country_agg['victims_avg'].fillna(0)))

print(f'Countries matched to GeoJSON: {len(killers_dict)}')

In [ ]:
# ── Build Folium choropleth — killers count ───────────────────────────────────
m = folium.Map(
    location=[20, 0], zoom_start=2,
    tiles='CartoDB positron',
    min_zoom=1, max_zoom=8
)

# Choropleth layer
folium.Choropleth(
    geo_data=geojson_path,
    name='Serial Killers Count',
    data=country_agg,
    columns=['country_geo', 'killers'],
    key_on='feature.properties.name',
    fill_color='YlOrRd',
    fill_opacity=0.75,
    line_opacity=0.2,
    legend_name='Number of documented serial killers',
    nan_fill_color='#F5F5F0',
    nan_fill_opacity=0.4,
).add_to(m)

# ── Tooltip layer (hover details) ─────────────────────────────────────────────
with open(geojson_path) as f:
    geo = json.load(f)

for feature in geo['features']:
    name = feature['properties']['name']
    k    = killers_dict.get(name, 0)
    v    = victims_dict.get(name, 0)
    a    = avg_dict.get(name, 0)
    if k > 0:
        tooltip_text = (
            f"<b>{name}</b><br>"
            f"Killers: {k}<br>"
            f"Total victims: {int(v) if v else 'n/a'}<br>"
            f"Avg victims: {a:.1f}"
        )
        folium.GeoJson(
            feature,
            style_function=lambda x: {
                'fillColor': 'transparent',
                'color':     'transparent',
                'weight':    0,
            },
            tooltip=folium.Tooltip(tooltip_text, sticky=True)
        ).add_to(m)

# ── Title box ─────────────────────────────────────────────────────────────────
title_html = '''
<div style="position: fixed; top: 16px; left: 50%; transform: translateX(-50%);
            background: white; padding: 10px 20px; border-radius: 8px;
            font-family: sans-serif; font-size: 14px; font-weight: 500;
            box-shadow: 0 1px 6px rgba(0,0,0,0.15); z-index: 1000;">
    🔪 Documented Serial Killers by Country · 633 killers · 101 countries
</div>
'''
m.get_root().html.add_child(folium.Element(title_html))

folium.LayerControl().add_to(m)

m.save('serial_killers_map.html')
print('✓ Saved: serial_killers_map.html')
m  # display inline in Jupyter

In [ ]:
# ── Second map: lethality (total victims) ─────────────────────────────────────
m2 = folium.Map(
    location=[20, 0], zoom_start=2,
    tiles='CartoDB positron'
)

folium.Choropleth(
    geo_data=geojson_path,
    name='Total Victims',
    data=country_agg.dropna(subset=['victims_total']),
    columns=['country_geo', 'victims_total'],
    key_on='feature.properties.name',
    fill_color='Reds',
    fill_opacity=0.75,
    line_opacity=0.2,
    legend_name='Total documented proven victims',
    nan_fill_color='#F5F5F0',
    nan_fill_opacity=0.4,
).add_to(m2)

title2 = '''
<div style="position: fixed; top: 16px; left: 50%; transform: translateX(-50%);
            background: white; padding: 10px 20px; border-radius: 8px;
            font-family: sans-serif; font-size: 14px; font-weight: 500;
            box-shadow: 0 1px 6px rgba(0,0,0,0.15); z-index: 1000;">
    🔪 Total Documented Victims by Country
</div>
'''
m2.get_root().html.add_child(folium.Element(title2))

m2.save('serial_killers_victims_map.html')
print('✓ Saved: serial_killers_victims_map.html')
m2

---
## 4. Decade-by-Decade Breakdown — How the Map Changed

In [ ]:
# ── Static: top countries per decade ──────────────────────────────────────────
df_mod = df[df['year_start'] >= 1940].dropna(subset=['decade']).copy()
df_mod['decade'] = df_mod['decade'].astype(int)
df_mod = df_mod[df_mod['decade'] <= 2010]

decades = sorted(df_mod['decade'].unique())

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Top 5 Countries per Decade (1940–2010)', fontsize=14, fontweight='bold')
axes = axes.flatten()

for i, decade in enumerate(decades[:8]):
    ax = axes[i]
    subset = df_mod[df_mod['decade'] == decade]
    top5   = subset['country'].value_counts().head(5)

    bar_c = [C['red'] if c == 'United States' else C['blue']
             for c in top5.index]

    bars = ax.barh(top5.index[::-1], top5.values[::-1],
                   color=bar_c[::-1], zorder=3, height=0.6)
    ax.set_title(f'{decade}s  (n={len(subset)})', fontsize=11, pad=6)
    ax.bar_label(bars, padding=2, fontsize=8, color=C['gray'])
    ax.set_xlim(0, top5.max() * 1.35)
    ax.tick_params(labelsize=9)
    ax.spines['bottom'].set_visible(False)

plt.tight_layout()
plt.savefig('02e_top_countries_by_decade.png')
plt.show()

In [ ]:
# ── Stacked area: continent activity over time ────────────────────────────────
df_area = df_mod[df_mod['continent'] != 'Other'].copy()

area_pivot = (
    df_area.groupby(['decade', 'continent'])
    .size()
    .unstack(fill_value=0)
)

cont_order   = area_pivot.sum().sort_values(ascending=False).index.tolist()
area_ordered = area_pivot[cont_order]

cont_colors_list = [C['red'], C['blue'], C['teal'], C['orange'],
                    C['purple'], C['green'], C['gray']]

fig, ax = plt.subplots(figsize=(12, 6))
area_ordered.plot.area(
    ax=ax,
    color=cont_colors_list[:len(cont_order)],
    alpha=0.85
)

ax.set_xlabel('Decade')
ax.set_ylabel('Number of killers')
ax.set_title('Serial Killer Activity by Continent Over Time (1940–2010)', pad=14)
ax.set_xticks(decades)
ax.set_xticklabels([f"{d}s" for d in decades], rotation=0)
ax.legend(loc='upper left', fontsize=9, title='Continent')

plt.tight_layout()
plt.savefig('02f_continent_area_chart.png')
plt.show()

---
## 5. Key Geographic Findings

In [ ]:
top_continent  = continent_stats['killers'].idxmax()
top_lethal_c   = continent_stats['victims_avg'].idxmax()
top_percapita  = df_cap.sort_values('killers_per_10m', ascending=False).iloc[0]
top_victims_c  = country_agg.sort_values('victims_total', ascending=False).iloc[0]
top_avg_c      = country_agg.dropna(subset=['victims_avg']).sort_values(
                   'victims_avg', ascending=False).iloc[0]

print('═' * 62)
print('  MODULE 02: GEOGRAPHIC ANALYSIS — KEY FINDINGS')
print('═' * 62)
print(f'\n  CONTINENT LEVEL')
print(f'  • Most killers:      {top_continent} ({continent_stats.loc[top_continent,"killers"]:.0f} cases)')
print(f'  • Most lethal avg:   {top_lethal_c} ({continent_stats.loc[top_lethal_c,"victims_avg"]:.1f} avg victims)')
print(f'\n  COUNTRY LEVEL')
print(f'  • Most killers:      United States (95)')
print(f'  • Most total victims:{top_victims_c["country"]} ({int(top_victims_c["victims_total"])} proven deaths)')
print(f'  • Most lethal avg:   {top_avg_c["country"]} ({top_avg_c["victims_avg"]:.1f} avg per killer)')
print(f'  • Per capita leader: {top_percapita["country"]} ({top_percapita["killers_per_10m"]:.1f} per 10M pop)')
print(f'\n  TEMPORAL')
print(f'  • Europe surpassed Americas in 1990s–2000s documentation')
print(f'  • Asia rising: India, South Korea, Iran appear strongly post-1970')
print(f'  • Africa remains underrepresented — documentation bias likely')
print(f'\n  MAPS SAVED')
print(f'  • serial_killers_map.html         (killers count choropleth)')
print(f'  • serial_killers_victims_map.html  (total victims choropleth)')
print(f'\n  ── NEXT: Module 03 → Predictive model (Random Forest + SHAP)')
print('═' * 62)

---

## What's Next

| Module | Focus | Techniques |
|--------|-------|------------|
| **03** | Predictive modelling | Random Forest, XGBoost, SHAP feature importance |
| **04** | Storytelling output | LinkedIn carousel, portfolio slides |

---
*Portfolio project · [Natalia Prokofyeva] · 2026*

---

<div style='background:#1A1A2E;padding:20px 24px;border-radius:8px;border-left:6px solid #C0392B;margin:20px 0'>
<span style='color:#C0392B;font-size:13px;font-weight:600;letter-spacing:2px'>MODULE 03</span><br>
<span style='color:white;font-size:22px;font-weight:700'>Predictive Modelling</span><br>
<span style='color:#AAAAAA;font-size:13px'>Random Forest · feature importance · partial effects · model comparison</span>
</div>

---


## 0. Setup

In [ ]:
# Install if needed — all come with Anaconda
# !pip install scikit-learn shap

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
from sklearn.inspection import permutation_importance

# ── Visual style ──────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor':  'white', 'axes.facecolor':    'white',
    'axes.spines.top':   False,   'axes.spines.right': False,
    'axes.spines.left':  False,   'axes.grid':         True,
    'grid.color':        '#EEEEEE','grid.linewidth':    0.6,
    'axes.axisbelow':    True,    'font.family':       'sans-serif',
    'font.size':         11,      'axes.titlesize':    14,
    'axes.titleweight':  'bold',  'legend.frameon':    False,
    'figure.dpi':        120,     'savefig.dpi':       150,
    'savefig.bbox':      'tight',
})

C = {
    'primary': '#1A1A2E', 'red':    '#C0392B', 'red_light': '#E74C3C',
    'blue':    '#2980B9', 'gray':   '#7F8C8D', 'green':     '#27AE60',
    'orange':  '#E67E22', 'purple': '#8E44AD', 'teal':      '#16A085',
}

SEED = 42
print('Setup complete.')

---
## 1. Load & Engineer Features

In [ ]:
df = pd.read_csv('serial_killers_master.csv')

for col in ['victims_proven', 'victims_possible', 'victims_max',
            'year_start', 'year_end', 'years_span', 'decade']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# ── Continent mapping ─────────────────────────────────────────────────────────
continent_map = {
    'United States':'North America','Canada':'North America','Mexico':'North America',
    'Colombia':'South America','Brazil':'South America','Argentina':'South America',
    'Peru':'South America','Venezuela':'South America','Bolivia':'South America',
    'Russia':'Europe','Italy':'Europe','Poland':'Europe','Ukraine':'Europe',
    'United Kingdom':'Europe','Belgium':'Europe','Germany':'Europe','France':'Europe',
    'Spain':'Europe','Austria':'Europe','Netherlands':'Europe','Greece':'Europe',
    'Hungary':'Europe','Romania':'Europe','Finland':'Europe','Sweden':'Europe',
    'Norway':'Europe','Denmark':'Europe','Czech Republic':'Europe','Slovakia':'Europe',
    'Latvia':'Europe','Estonia':'Europe','Lithuania':'Europe','Belarus':'Europe',
    'Serbia':'Europe','Croatia':'Europe','Slovenia':'Europe','Portugal':'Europe',
    'Switzerland':'Europe','Bulgaria':'Europe','Moldova':'Europe','Ireland':'Europe',
    'North Macedonia':'Europe','Malta':'Europe','Cyprus':'Europe','Iceland':'Europe',
    'Kazakhstan':'Asia','Iran':'Asia','India':'Asia','China':'Asia',
    'South Korea':'Asia','Japan':'Asia','Indonesia':'Asia','Thailand':'Asia',
    'Malaysia':'Asia','Philippines':'Asia','Pakistan':'Asia','Bangladesh':'Asia',
    'Taiwan':'Asia','Singapore':'Asia','Turkey':'Asia','Israel':'Asia',
    'Iraq':'Asia','Jordan':'Asia','Saudi Arabia':'Asia','Afghanistan':'Asia',
    'Yemen':'Asia','Vietnam':'Asia','Uzbekistan':'Asia','Kyrgyzstan':'Asia',
    'South Africa':'Africa','Nigeria':'Africa','Egypt':'Africa','Morocco':'Africa',
    'Zimbabwe':'Africa','Zambia':'Africa','Rwanda':'Africa','Ghana':'Africa',
    'Algeria':'Africa','Tunisia':'Africa','Namibia':'Africa','Eswatini':'Africa',
    'Australian':'Oceania','New Zealand':'Oceania','Fiji':'Oceania',
    'Australia':'Oceania',
}
df['continent'] = df['country'].map(continent_map).fillna('Other')

# ── Feature engineering ───────────────────────────────────────────────────────
df['us_flag']       = (df['country'] == 'United States').astype(int)
df['europe_flag']   = (df['continent'] == 'Europe').astype(int)
df['asia_flag']     = (df['continent'] == 'Asia').astype(int)
df['post_1970']     = ((df['year_start'] >= 1970) & df['year_start'].notna()).astype(int)
df['post_1990']     = ((df['year_start'] >= 1990) & df['year_start'].notna()).astype(int)
df['is_poison']     = (df['method'] == 'Poisoning').astype(int)
df['is_strang']     = (df['method'] == 'Strangulation').astype(int)
df['is_shoot']      = (df['method'] == 'Shooting').astype(int)
df['is_male']       = (df['gender'].str.strip().str.title() == 'Male').astype(int)
df['is_executed']   = (df['status'] == 'Executed').astype(int)
df['is_life']       = (df['status'] == 'Life imprisonment').astype(int)
df['decade_norm']   = (df['decade'] - 1900) / 100   # normalise decade

print(f'Dataset: {len(df):,} rows')
print(f'Features engineered: us_flag, europe_flag, asia_flag, post_1970, post_1990,')
print(f'                     is_poison, is_strang, is_shoot, is_male, is_executed, is_life, decade_norm')

---
## 2. Model A — Predicting Victim Count

> **Question:** Given what we know about a killer (geography, era, method), can we predict how many victims they'll claim?

**Target:** `victims_proven` (331 rows with non-null values)

In [ ]:
# ── Prepare model A data ──────────────────────────────────────────────────────
FEATURES_A = [
    'us_flag', 'europe_flag', 'asia_flag',
    'post_1970', 'post_1990', 'decade_norm',
    'is_poison', 'is_strang', 'is_shoot',
    'is_male',
]
TARGET_A = 'victims_proven'

df_a = df.dropna(subset=[TARGET_A, 'year_start']).copy()
df_a = df_a[df_a[TARGET_A] >= 2]  # FBI definition: 2+ victims

X_a = df_a[FEATURES_A].fillna(0)
y_a = df_a[TARGET_A]

# Log-transform target (right-skewed distribution)
y_a_log = np.log1p(y_a)

X_train_a, X_test_a, y_train_a, y_test_a = train_test_split(
    X_a, y_a_log, test_size=0.2, random_state=SEED
)

print(f'Training set:  {len(X_train_a)} rows')
print(f'Test set:      {len(X_test_a)} rows')
print(f'Target range:  {y_a.min():.0f} – {y_a.max():.0f} victims')
print(f'Target (log):  {y_a_log.min():.2f} – {y_a_log.max():.2f}')

In [ ]:
# ── Train Random Forest A ─────────────────────────────────────────────────────
rf_a = RandomForestRegressor(
    n_estimators=300,
    max_depth=6,
    min_samples_leaf=5,
    random_state=SEED,
    n_jobs=-1
)
rf_a.fit(X_train_a, y_train_a)

# ── Evaluate ──────────────────────────────────────────────────────────────────
y_pred_log = rf_a.predict(X_test_a)
y_pred     = np.expm1(y_pred_log)   # back to original scale
y_true     = np.expm1(y_test_a)

mae_a  = mean_absolute_error(y_true, y_pred)
r2_a   = r2_score(y_test_a, y_pred_log)   # R² on log scale

# Cross-validation
cv_scores = cross_val_score(rf_a, X_a, y_a_log, cv=5, scoring='r2')

print(f'── Model A: Predict victim count ────────────────────────────')
print(f'R² (test, log scale):      {r2_a:.3f}')
print(f'MAE (original scale):      {mae_a:.1f} victims')
print(f'Cross-val R² (5-fold):     {cv_scores.mean():.3f} ± {cv_scores.std():.3f}')
print(f'\nInterpretation:')
print(f'  R² = {r2_a:.2f} means the model explains {r2_a*100:.0f}% of variance in log(victims).')
print(f'  MAE = {mae_a:.1f} means predictions are off by ~{mae_a:.0f} victims on average.')
print(f'  With only {len(df_a)} rows, this is directionally meaningful, not production-ready.')

In [ ]:
# ── Feature importance — Model A ──────────────────────────────────────────────
feat_imp_a = pd.Series(
    rf_a.feature_importances_,
    index=FEATURES_A
).sort_values(ascending=True)

# Human-readable feature labels
label_map = {
    'us_flag':       'Killer is from USA',
    'europe_flag':   'Killer is from Europe',
    'asia_flag':     'Killer is from Asia',
    'post_1970':     'Active after 1970',
    'post_1990':     'Active after 1990',
    'decade_norm':   'Era (decade, normalised)',
    'is_poison':     'Method: Poisoning',
    'is_strang':     'Method: Strangulation',
    'is_shoot':      'Method: Shooting',
    'is_male':       'Gender: Male',
    'is_executed':   'Outcome: Executed',
    'is_life':       'Outcome: Life sentence',
}
feat_imp_a.index = [label_map.get(f, f) for f in feat_imp_a.index]

fig, ax = plt.subplots(figsize=(10, 5))
colors = [C['red'] if v == feat_imp_a.max() else
          C['blue'] if v >= feat_imp_a.quantile(0.6) else
          C['gray'] for v in feat_imp_a.values]

bars = ax.barh(feat_imp_a.index, feat_imp_a.values, color=colors, zorder=3)
ax.set_xlabel('Feature importance (mean decrease in impurity)')
ax.set_title('Model A: What Predicts Victim Count?', pad=14)
ax.bar_label(bars, fmt='%.3f', padding=3, fontsize=9, color=C['gray'])
ax.set_xlim(0, feat_imp_a.max() * 1.22)

# Annotation
top_feat = feat_imp_a.idxmax()
ax.text(0.98, 0.04,
        f'Top predictor: {top_feat}',
        transform=ax.transAxes, ha='right', va='bottom',
        fontsize=9, color=C['primary'], style='italic',
        bbox=dict(boxstyle='round,pad=0.3', fc='white', ec=C['gray'], lw=0.8))

plt.tight_layout()
plt.savefig('03a_feature_importance_victims.png')
plt.show()

In [ ]:
# ── Predicted vs Actual scatter ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Model A: Predicted vs Actual Victim Count', fontsize=14, fontweight='bold')

# Scatter
ax = axes[0]
ax.scatter(y_true, y_pred, alpha=0.45, s=30,
           color=C['blue'], edgecolors=C['primary'], linewidths=0.3)
lim = max(y_true.max(), y_pred.max()) * 1.05
ax.plot([0, lim], [0, lim], color=C['red'], lw=1.5, ls='--', label='Perfect prediction')
ax.set_xlabel('Actual victims')
ax.set_ylabel('Predicted victims')
ax.set_title(f'Scatter (R² = {r2_a:.2f})')
ax.legend()

# Residuals
ax = axes[1]
residuals = y_true - y_pred
ax.hist(residuals, bins=30, color=C['red'], edgecolor='white', linewidth=0.4, zorder=3)
ax.axvline(0, color=C['primary'], lw=1.5, ls='--', label='Zero error')
ax.set_xlabel('Residual (actual − predicted)')
ax.set_ylabel('Count')
ax.set_title('Residual distribution')
ax.legend()
ax.text(0.97, 0.94, f'MAE = {mae_a:.1f} victims',
        transform=ax.transAxes, ha='right', va='top', fontsize=9, color=C['primary'],
        bbox=dict(boxstyle='round,pad=0.3', fc='white', ec=C['gray'], lw=0.8))

plt.tight_layout()
plt.savefig('03b_predicted_vs_actual_victims.png')
plt.show()

---
## 3. Model B — Predicting Years Active Before Capture

> **Question:** What determines how long a killer operates before being caught?

**Target:** `years_span` (606 rows — much better coverage)

In [ ]:
# ── Prepare model B ───────────────────────────────────────────────────────────
FEATURES_B = [
    'us_flag', 'europe_flag', 'asia_flag',
    'post_1970', 'post_1990', 'decade_norm',
    'is_poison', 'is_strang', 'is_shoot',
    'is_male', 'is_executed', 'is_life',
]
TARGET_B = 'years_span'

df_b = df.dropna(subset=[TARGET_B, 'year_start']).copy()
df_b = df_b[(df_b[TARGET_B] >= 0) & (df_b[TARGET_B] <= 50)]  # remove extreme outliers

X_b = df_b[FEATURES_B].fillna(0)
y_b = df_b[TARGET_B]
y_b_log = np.log1p(y_b)  # log-transform (many 0s and 1s)

X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(
    X_b, y_b_log, test_size=0.2, random_state=SEED
)

print(f'Training set:  {len(X_train_b)} rows')
print(f'Test set:      {len(X_test_b)} rows')
print(f'Target range:  {y_b.min():.0f} – {y_b.max():.0f} years')
print(f'Target median: {y_b.median():.0f} years')

In [ ]:
# ── Train Random Forest B ─────────────────────────────────────────────────────
rf_b = RandomForestRegressor(
    n_estimators=300,
    max_depth=6,
    min_samples_leaf=5,
    random_state=SEED,
    n_jobs=-1
)
rf_b.fit(X_train_b, y_train_b)

y_pred_b_log = rf_b.predict(X_test_b)
y_pred_b     = np.expm1(y_pred_b_log)
y_true_b     = np.expm1(y_test_b)

mae_b = mean_absolute_error(y_true_b, y_pred_b)
r2_b  = r2_score(y_test_b, y_pred_b_log)
cv_b  = cross_val_score(rf_b, X_b, y_b_log, cv=5, scoring='r2')

print(f'── Model B: Predict years to capture ────────────────────────')
print(f'R² (test, log scale):      {r2_b:.3f}')
print(f'MAE (original scale):      {mae_b:.1f} years')
print(f'Cross-val R² (5-fold):     {cv_b.mean():.3f} ± {cv_b.std():.3f}')
print(f'\nInterpretation:')
print(f'  MAE = {mae_b:.1f} years means the model is off by ~{mae_b:.0f} years on average.')
print(f'  Given median of {y_b.median():.0f} years, this is {'decent' if mae_b < y_b.median() else 'modest'} performance.')

In [ ]:
# ── Feature importance — Model B ──────────────────────────────────────────────
feat_imp_b = pd.Series(
    rf_b.feature_importances_,
    index=FEATURES_B
).sort_values(ascending=True)
feat_imp_b.index = [label_map.get(f, f) for f in feat_imp_b.index]

fig, ax = plt.subplots(figsize=(10, 5))
colors_b = [C['red'] if v == feat_imp_b.max() else
            C['blue'] if v >= feat_imp_b.quantile(0.6) else
            C['gray'] for v in feat_imp_b.values]

bars = ax.barh(feat_imp_b.index, feat_imp_b.values, color=colors_b, zorder=3)
ax.set_xlabel('Feature importance (mean decrease in impurity)')
ax.set_title('Model B: What Predicts Years Active Before Capture?', pad=14)
ax.bar_label(bars, fmt='%.3f', padding=3, fontsize=9, color=C['gray'])
ax.set_xlim(0, feat_imp_b.max() * 1.22)

top_b = feat_imp_b.idxmax()
ax.text(0.98, 0.04,
        f'Top predictor: {top_b}',
        transform=ax.transAxes, ha='right', va='bottom',
        fontsize=9, color=C['primary'], style='italic',
        bbox=dict(boxstyle='round,pad=0.3', fc='white', ec=C['gray'], lw=0.8))

plt.tight_layout()
plt.savefig('03c_feature_importance_span.png')
plt.show()

In [ ]:
# ── Predicted vs Actual — Model B ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Model B: Predicted vs Actual Years Active', fontsize=14, fontweight='bold')

ax = axes[0]
ax.scatter(y_true_b, y_pred_b, alpha=0.35, s=25,
           color=C['teal'], edgecolors=C['primary'], linewidths=0.3)
lim_b = max(y_true_b.max(), y_pred_b.max()) * 1.05
ax.plot([0, lim_b], [0, lim_b], color=C['red'], lw=1.5, ls='--', label='Perfect prediction')
ax.set_xlabel('Actual years active')
ax.set_ylabel('Predicted years active')
ax.set_title(f'Scatter (R² = {r2_b:.2f})')
ax.legend()

ax = axes[1]
res_b = y_true_b - y_pred_b
ax.hist(res_b, bins=30, color=C['teal'], edgecolor='white', linewidth=0.4, zorder=3)
ax.axvline(0, color=C['primary'], lw=1.5, ls='--')
ax.set_xlabel('Residual (actual − predicted)')
ax.set_ylabel('Count')
ax.set_title('Residual distribution')
ax.text(0.97, 0.94, f'MAE = {mae_b:.1f} years',
        transform=ax.transAxes, ha='right', va='top', fontsize=9, color=C['primary'],
        bbox=dict(boxstyle='round,pad=0.3', fc='white', ec=C['gray'], lw=0.8))

plt.tight_layout()
plt.savefig('03d_predicted_vs_actual_span.png')
plt.show()

---
## 4. Side-by-Side Feature Comparison

> Which features matter for **lethality** vs **evasion**? Are they the same story?

In [ ]:
# ── Compare importances: Model A vs Model B ───────────────────────────────────
# Align features (B has 2 extra)
common_feats = [f for f in FEATURES_A if f in FEATURES_B]
imp_a = pd.Series(rf_a.feature_importances_, index=FEATURES_A)
imp_b = pd.Series(rf_b.feature_importances_, index=FEATURES_B)

compare = pd.DataFrame({
    'Victim count': imp_a[common_feats],
    'Years active':  imp_b[common_feats],
})
compare.index = [label_map.get(f, f) for f in compare.index]
compare = compare.sort_values('Victim count', ascending=True)

fig, ax = plt.subplots(figsize=(11, 6))

x     = np.arange(len(compare))
w     = 0.38
bars1 = ax.barh(x - w/2, compare['Victim count'], height=w,
                color=C['red'],  label='Model A: Victim count', zorder=3, alpha=0.9)
bars2 = ax.barh(x + w/2, compare['Years active'],  height=w,
                color=C['teal'], label='Model B: Years active',  zorder=3, alpha=0.9)

ax.set_yticks(x)
ax.set_yticklabels(compare.index)
ax.set_xlabel('Feature importance')
ax.set_title('Feature Importance: Lethality vs Evasion — Same Drivers?', pad=14)
ax.legend()

plt.tight_layout()
plt.savefig('03e_importance_comparison.png')
plt.show()

# Find divergences
compare['diff'] = (compare['Victim count'] - compare['Years active']).abs()
biggest_diff = compare['diff'].idxmax()
print(f'Biggest divergence between models: {biggest_diff}')
print(compare[['Victim count', 'Years active', 'diff']].sort_values('diff', ascending=False).to_string())

---
## 5. Partial Dependence Plots — How Each Feature Affects Outcome

In [ ]:
# ── Manual partial dependence for key binary features ─────────────────────────
# For each binary feature: compare avg predicted outcome when feature = 0 vs 1

def partial_effect(model, X_data, feature, values=[0, 1]):
    """Compute mean prediction when feature is fixed at each value."""
    results = []
    for val in values:
        X_copy = X_data.copy()
        X_copy[feature] = val
        pred = np.expm1(model.predict(X_copy))
        results.append(pred.mean())
    return results

binary_features = [
    ('us_flag',    'USA vs other'),
    ('europe_flag','Europe vs other'),
    ('asia_flag',  'Asia vs other'),
    ('post_1970',  'After 1970 vs before'),
    ('post_1990',  'After 1990 vs before'),
    ('is_poison',  'Poison vs other method'),
    ('is_male',    'Male vs female'),
]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Partial Effects: How Each Factor Shifts Expected Victim Count',
             fontsize=14, fontweight='bold')
axes = axes.flatten()

for i, (feat, label) in enumerate(binary_features):
    if feat not in X_a.columns:
        continue
    ax = axes[i]
    effects = partial_effect(rf_a, X_a, feat)
    bar_c   = [C['gray'], C['red'] if effects[1] > effects[0] else C['blue']]
    bars    = ax.bar(['No', 'Yes'], effects, color=bar_c, width=0.5, zorder=3)
    ax.set_title(label, fontsize=10, pad=6)
    ax.set_ylabel('Predicted avg victims')
    ax.bar_label(bars, fmt='%.1f', padding=3, fontsize=9, color=C['gray'])
    ax.set_ylim(0, max(effects) * 1.3)

# Hide unused subplot
axes[-1].set_visible(False)

plt.tight_layout()
plt.savefig('03f_partial_effects.png')
plt.show()

---
## 6. Gradient Boosting — Does a Stronger Model Help?

In [ ]:
# ── Compare RF vs GBM on Model A (victims) ────────────────────────────────────
gbm_a = GradientBoostingRegressor(
    n_estimators=300, max_depth=4, learning_rate=0.05,
    min_samples_leaf=5, random_state=SEED
)
gbm_a.fit(X_train_a, y_train_a)

cv_rf  = cross_val_score(rf_a,  X_a, y_a_log, cv=5, scoring='r2')
cv_gbm = cross_val_score(gbm_a, X_a, y_a_log, cv=5, scoring='r2')

fig, ax = plt.subplots(figsize=(9, 4))

models = ['Random Forest', 'Gradient Boosting']
means  = [cv_rf.mean(), cv_gbm.mean()]
stds   = [cv_rf.std(),  cv_gbm.std()]

colors_m = [C['red'], C['blue']]
bars = ax.bar(models, means, color=colors_m, width=0.45, zorder=3)
ax.errorbar(models, means, yerr=stds, fmt='none',
            color=C['primary'], capsize=6, lw=1.5, zorder=4)
ax.set_ylabel('Cross-validated R² (5-fold)')
ax.set_title('Model A: Random Forest vs Gradient Boosting', pad=12)
ax.set_ylim(0, max(means) * 1.4)
ax.bar_label(bars, fmt='%.3f', padding=5, fontsize=10, color=C['gray'])

winner = models[np.argmax(means)]
ax.text(0.98, 0.95, f'Better model: {winner}',
        transform=ax.transAxes, ha='right', va='top', fontsize=9, color=C['primary'],
        bbox=dict(boxstyle='round,pad=0.3', fc='white', ec=C['gray'], lw=0.8))

plt.tight_layout()
plt.savefig('03g_model_comparison.png')
plt.show()

print(f'Random Forest CV R²:      {cv_rf.mean():.3f} ± {cv_rf.std():.3f}')
print(f'Gradient Boosting CV R²:  {cv_gbm.mean():.3f} ± {cv_gbm.std():.3f}')
print(f'Winner: {winner}')

---
## 7. Key Findings — Module 03

In [ ]:
# ── Auto-compute top features ─────────────────────────────────────────────────
imp_a_named = pd.Series(rf_a.feature_importances_, index=FEATURES_A)
imp_b_named = pd.Series(rf_b.feature_importances_, index=FEATURES_B)
top_a = label_map.get(imp_a_named.idxmax(), imp_a_named.idxmax())
top_b = label_map.get(imp_b_named.idxmax(), imp_b_named.idxmax())

print('═' * 62)
print('  MODULE 03: PREDICTIVE MODELLING — KEY FINDINGS')
print('═' * 62)
print(f'\n  MODEL A — Predict victim count')
print(f'  • Algorithm:      Random Forest (300 trees)')
print(f'  • Training rows:  {len(X_train_a)}')
print(f'  • R² (test):      {r2_a:.2f} (log scale)')
print(f'  • MAE:            {mae_a:.1f} victims')
print(f'  • CV R² (5-fold): {cv_rf.mean():.2f} ± {cv_rf.std():.2f}')
print(f'  • Top predictor:  {top_a}')
print(f'\n  MODEL B — Predict years active before capture')
print(f'  • Training rows:  {len(X_train_b)}')
print(f'  • R² (test):      {r2_b:.2f} (log scale)')
print(f'  • MAE:            {mae_b:.1f} years')
print(f'  • CV R² (5-fold): {cv_b.mean():.2f} ± {cv_b.std():.2f}')
print(f'  • Top predictor:  {top_b}')
print(f'\n  HONEST ASSESSMENT')
print(f'  • 633 killers is small for ML — models are exploratory')
print(f'  • Value is in feature importance, not prediction accuracy')
print(f'  • Era (decade) and geography dominate both models')
print(f'  • Method data only available for ~20% of dataset')
print(f'\n  ── NEXT: Module 04 → LinkedIn slides & portfolio output')
print('═' * 62)